# AlibabaCloud Lingjiang 2.0 — GIFT-Eval

## Environment preparation

Use Python 3.10 or newer. From the repository root, install the benchmark package and the Alibaba Cloud SDK used by this notebook:

```bash
python -m pip install -e .
python -m pip install alibabacloud_brain_industrial20200920==2.1.2
```

Download the GIFT-Eval evaluation dataset as described in the repository README, then set `GIFT_EVAL` to its local directory before starting the notebook. Enter an authorized Alibaba Cloud AccessKey, SecretKey, and AICS service ID in the next cell. Keep these credentials out of any published notebook. The service ID must point to the existing forwarding canvas that supports `transport_v2`. Run the notebook from the repository root so its results are written under `results/lingjiang2.0-evaluation/`.

## Evaluation

Set `GIFT_EVAL` to the downloaded dataset directory before running. This notebook runs all 97 official GIFT-Eval configurations through the existing Alibaba Cloud forwarding canvas. It sends complete historical arrays in resumable chunks, with each AICS call under 900,000 bytes. Each completed batch and configuration is saved so a restarted run continues from the last committed point.


In [ ]:
# Set AK/SK
access_key = "***"
secret_key = "***"
service_id = "***"

Set all the dataset to evaluation

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()
dataset_root = os.getenv('GIFT_EVAL')
assert dataset_root and os.path.isdir(dataset_root), 'Set GIFT_EVAL to the downloaded GIFT-Eval dataset directory'
os.environ['GIFT_EVAL'] = dataset_root

short_datasets = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

# Only these 21 configurations are evaluated on the medium and long terms.
med_long_datasets = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

dataset_properties_map = {
    "bitbrains_fast_storage": {"domain": "Web/CloudOps", "frequency": "H", "num_variates": 2},
    "bitbrains_rnd": {"domain": "Web/CloudOps", "frequency": "H", "num_variates": 2},
    "bizitobs_application": {"domain": "Web/CloudOps", "frequency": "10S", "num_variates": 2},
    "bizitobs_l2c": {"domain": "Web/CloudOps", "frequency": "H", "num_variates": 7},
    "bizitobs_service": {"domain": "Web/CloudOps", "frequency": "10S", "num_variates": 2},
    "car_parts": {"domain": "Sales", "frequency": "M", "num_variates": 1},
    "covid_deaths": {"domain": "Healthcare", "frequency": "D", "num_variates": 1},
    "electricity": {"domain": "Energy", "frequency": "W", "num_variates": 1},
    "ett1": {"domain": "Energy", "frequency": "W", "num_variates": 7},
    "ett2": {"domain": "Energy", "frequency": "W", "num_variates": 7},
    "hierarchical_sales": {"domain": "Sales", "frequency": "W-WED", "num_variates": 1},
    "hospital": {"domain": "Healthcare", "frequency": "M", "num_variates": 1},
    "jena_weather": {"domain": "Nature", "frequency": "D", "num_variates": 21},
    "kdd_cup_2018": {"domain": "Nature", "frequency": "D", "num_variates": 1},
    "loop_seattle": {"domain": "Transport", "frequency": "D", "num_variates": 1},
    "m4_daily": {"domain": "Econ/Fin", "frequency": "D", "num_variates": 1},
    "m4_hourly": {"domain": "Econ/Fin", "frequency": "H", "num_variates": 1},
    "m4_monthly": {"domain": "Econ/Fin", "frequency": "M", "num_variates": 1},
    "m4_quarterly": {"domain": "Econ/Fin", "frequency": "Q", "num_variates": 1},
    "m4_weekly": {"domain": "Econ/Fin", "frequency": "W", "num_variates": 1},
    "m4_yearly": {"domain": "Econ/Fin", "frequency": "A", "num_variates": 1},
    "m_dense": {"domain": "Transport", "frequency": "D", "num_variates": 1},
    "restaurant": {"domain": "Sales", "frequency": "D", "num_variates": 1},
    "saugeen": {"domain": "Nature", "frequency": "M", "num_variates": 1},
    "solar": {"domain": "Energy", "frequency": "W", "num_variates": 1},
    "sz_taxi": {"domain": "Transport", "frequency": "H", "num_variates": 1},
    "temperature_rain": {"domain": "Nature", "frequency": "D", "num_variates": 1},
    "us_births": {"domain": "Healthcare", "frequency": "M", "num_variates": 1},
}


def configurations():
    """The canonical 97 (dataset, term) pairs, keyed as base/freq/term."""
    result = []
    for name in sorted(set(short_datasets.split()) | set(med_long_datasets.split())):
        base = name.split('/')[0].lower()
        base = pretty_names.get(base, base)
        freq = name.split('/')[1] if '/' in name else dataset_properties_map[base]['frequency']
        for term in ('short', 'medium', 'long'):
            if term != 'short' and name not in med_long_datasets.split():
                continue
            result.append((name, term, f'{base}/{freq}/{term}', freq, dataset_properties_map[base]))
    return result


print(len(configurations()), 'configurations')

Set the metric functions

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)


def metrics():
    return [
        MSE(forecast_type='mean'),
        MSE(forecast_type=0.5),
        MAE(),
        MASE(),
        MAPE(),
        SMAPE(),
        MSIS(),
        RMSE(),
        NRMSE(),
        ND(),
        MeanWeightedSumQuantileLoss(quantile_levels=[x / 10 for x in range(1, 10)]),
    ]


KEYS = ['MSE[mean]', 'MSE[0.5]', 'MAE[0.5]', 'MASE[0.5]', 'MAPE[0.5]',
        'sMAPE[0.5]', 'MSIS', 'RMSE[mean]', 'NRMSE[mean]', 'ND[0.5]',
        'mean_weighted_sum_quantile_loss']
HEADER = ['dataset', 'model'] + ['eval_metrics/' + k for k in KEYS] + ['domain', 'num_variates']

Model name and result saving path

In [ ]:
from pathlib import Path

# Written to the `model` column, matching the delivery reference CSV.
model_name = "Lingjiang2.0"
output_dir = Path("results/lingjiang2.0-evaluation")
output_dir.mkdir(parents=True, exist_ok=True)
csv_file_path = output_dir / "all_results.csv"

The existing Alibaba Cloud AICS service ID and public canvas remain unchanged. Each control call uses a fresh AICS job ID; a persistent logical request ID makes chunk uploads, inference, and result fetch resumable.


In [ ]:
import base64
import hashlib
import io
import lzma
import time
import uuid
import numpy as np
from alibabacloud_brain_industrial20200920.client import Client as BrainIndustrialClient
from alibabacloud_brain_industrial20200920.models import AicsOpenApiInvokeRequest
from alibabacloud_tea_openapi.models import Config

MAX_BYTES = 256 * 1024 * 1024
MAX_AICS_REQUEST_BYTES = 900_000  # Leave room for the SDK envelope below 1 MB.
FAILED_STATUSES = {'FAIL', 'FAILED', 'FAILURE', 'CANCELLED', 'CANCELED', 'ERROR', 'STOPPED'}


# The four codec functions define the wire format and must stay byte-compatible
# with the serving side. Arrays travel as base64 .npy, so float32 is lossless and
# missing values survive as NaN.
def encode(value):
    return base64.b64encode(lzma.compress(json.dumps(value, allow_nan=False).encode())).decode()


def decode(value):
    dec = lzma.LZMADecompressor()
    raw = dec.decompress(base64.b64decode(value, validate=True), max_length=MAX_BYTES + 1)
    if len(raw) > MAX_BYTES or not dec.eof:
        raise ValueError('Payload exceeds decompression limit or is incomplete')
    return json.loads(raw)


def pack_array(value):
    if value is None:
        return None
    buffer = io.BytesIO()
    np.save(buffer, np.asarray(value, dtype=np.float32), allow_pickle=False)
    return base64.b64encode(buffer.getvalue()).decode()


def unpack_array(value):
    if value is None:
        return None
    if not isinstance(value, str):
        raise ValueError('Expected base64 .npy array')
    arr = np.load(io.BytesIO(base64.b64decode(value, validate=True)), allow_pickle=False)
    if arr.dtype != np.float32 or arr.ndim not in (1, 2, 3) or arr.size == 0:
        raise ValueError('Expected nonempty float32 array with 1-3 dimensions')
    return arr


def result_payload(value):
    """Unwrap the AICS response envelope, whichever nesting the platform returns."""
    for _ in range(12):
        if isinstance(value, str):
            try:
                value = json.loads(value)
            except json.JSONDecodeError:
                return decode(value)
        elif isinstance(value, dict):
            if value.get('schema_version') == 2:
                return value
            if value.get('success') is False:
                raise RuntimeError('AICS invocation failed: '
                                   + str(value.get('message', value.get('code'))))
            for key in ('data', 'output', 'result', 're', 'value'):
                if key in value:
                    value = value[key]
                    break
            else:
                raise RuntimeError('Unrecognized AICS response envelope')
        else:
            raise RuntimeError('Empty or invalid AICS response')
    raise RuntimeError('AICS result envelope too deeply nested')


class PublicCloudClient:
    def __init__(self, service_id, access_key, secret_key, timeout=1800, poll_interval=3):
        config = Config(access_key_id=access_key, access_key_secret=secret_key,
                        region_id='cn-hangzhou', read_timeout=60000, connect_timeout=60000)
        config.endpoint = 'brain-industrial.cn-hangzhou.aliyuncs.com'
        self.client = BrainIndustrialClient(config)
        self.service_id, self.timeout, self.poll_interval = service_id, timeout, poll_interval
        self.calls, self.identities = 0, {}
        self.transport_calls, self.max_canvas_call_seconds = 0, 0.0
        self.cache_hits = 0
        self.run_id, self.cache_dir = None, None

    def _invoke(self, message, encoded_input=None):
        """One AICS job; retries use its job ID and poll without resending data."""
        value = encoded_input if encoded_input is not None else encode(message)
        params = {'input': value}
        wire_bytes = len(json.dumps(params, separators=(',', ':')).encode())
        if wire_bytes > MAX_AICS_REQUEST_BYTES:
            raise ValueError(f'AICS request is {wire_bytes} bytes; limit is {MAX_AICS_REQUEST_BYTES}')
        job_id = uuid.uuid4().hex
        deadline = time.monotonic() + self.timeout
        errors = 0
        while time.monotonic() < deadline:
            req = AicsOpenApiInvokeRequest(service_id=self.service_id, param=params,
                                           type='EXPERIMENT', job_id=job_id)
            started = time.monotonic()
            try:
                data = self.client.aics_open_api_invoke(req).body.data
                self.transport_calls += 1
                self.max_canvas_call_seconds = max(
                    self.max_canvas_call_seconds, time.monotonic() - started)
            except Exception as exc:
                self.transport_calls += 1
                self.max_canvas_call_seconds = max(
                    self.max_canvas_call_seconds, time.monotonic() - started)
                errors += 1
                retryable_status = str(getattr(exc, 'status_code', '')) in (
                    '429', '500', '502', '503', '504')
                if errors >= (12 if retryable_status else 3):
                    raise
                if retryable_status:
                    params = {'input': ''}
                time.sleep(min(30, self.poll_interval * 2 ** min(errors - 1, 3)))
                continue
            errors = 0
            if isinstance(data, str):
                data = json.loads(data)
            status = data.get('jobStatus')
            if status in FAILED_STATUSES:
                raise RuntimeError(f'AICS job {job_id} ended with {status}')
            if status in (None, 'SUCCESS'):
                for key in ('output', 'result'):
                    payload = data.get(key)
                    if payload not in (None, 'null', ''):
                        return result_payload(payload)
            params = {'input': ''}
            time.sleep(self.poll_interval)
        raise TimeoutError(f'AICS job {job_id} exceeded {self.timeout}s')

    @staticmethod
    def _control(reply, request_id):
        transport = reply.get('transport_v2') if isinstance(reply, dict) else None
        if not isinstance(transport, dict) or transport.get('request_id') != request_id:
            raise RuntimeError('Unexpected transport-v2 response')
        return transport

    def predict(self, request):
        if not self.run_id or self.cache_dir is None:
            raise RuntimeError('Set run_id and cache_dir before predicting')
        encoded_request = encode(request)
        digest = hashlib.sha256(encoded_request.encode('ascii')).hexdigest()
        request_id = hashlib.sha256((self.run_id + digest).encode()).hexdigest()[:32]
        cache_path = self.cache_dir / (request_id + '.json')
        if cache_path.exists():
            cached = json.loads(cache_path.read_text())
            if cached.get('digest') != digest:
                raise RuntimeError('Conflicting cached prediction')
            self.cache_hits += 1
            return cached['result']

        inline = {'transport_v2': 1, 'op': 'submit_inline', 'request_id': request_id,
                  'digest': digest, 'request': request}
        inline_encoded = encode(inline)
        inline_bytes = len(json.dumps({'input': inline_encoded}, separators=(',', ':')).encode())
        if inline_bytes <= MAX_AICS_REQUEST_BYTES:
            reply = self._invoke(inline, inline_encoded)
            self._control(reply, request_id)
        else:
            part_size = 400_000
            while True:
                pieces = [encoded_request[i:i + part_size]
                          for i in range(0, len(encoded_request), part_size)]
                uploads = [({'transport_v2': 1, 'op': 'upload', 'request_id': request_id,
                             'digest': digest, 'index': index, 'total': len(pieces),
                             'chunk': chunk}) for index, chunk in enumerate(pieces)]
                encoded_uploads = [encode(message) for message in uploads]
                if all(len(json.dumps({'input': value}, separators=(',', ':')).encode())
                       <= MAX_AICS_REQUEST_BYTES for value in encoded_uploads):
                    break
                part_size //= 2
                if part_size < 10_000:
                    raise ValueError('Cannot fit a transport chunk under the AICS limit')
            for message, value in zip(uploads, encoded_uploads):
                self._control(self._invoke(message, value), request_id)
            self._control(self._invoke({'transport_v2': 1, 'op': 'submit',
                                        'request_id': request_id, 'digest': digest}), request_id)

        deadline = time.monotonic() + self.timeout
        while time.monotonic() < deadline:
            status = self._control(self._invoke({'transport_v2': 1, 'op': 'status',
                                                 'request_id': request_id}), request_id)
            if status['state'] == 'done':
                chunks = []
                for index in range(status['result_parts']):
                    part = self._control(self._invoke({'transport_v2': 1, 'op': 'fetch',
                                                       'request_id': request_id,
                                                       'index': index}), request_id)
                    if part.get('index') != index:
                        raise RuntimeError('Result part index mismatch')
                    chunks.append(part['chunk'])
                encoded_result = ''.join(chunks)
                if hashlib.sha256(encoded_result.encode('ascii')).hexdigest() != status['result_digest']:
                    raise RuntimeError('Result integrity check failed')
                result = decode(encoded_result)
                self.cache_dir.mkdir(parents=True, exist_ok=True)
                temp = cache_path.with_suffix('.tmp')
                temp.write_text(json.dumps({'digest': digest, 'result': result}))
                temp.replace(cache_path)
                self.calls += 1
                return result
            if status['state'] == 'failed':
                raise RuntimeError(status.get('error', 'Remote inference failed'))
            time.sleep(self.poll_interval)
        raise TimeoutError(f'Transport request {request_id} exceeded {self.timeout}s')

client = PublicCloudClient(service_id, access_key, secret_key)

The notebook sends full observations and covariates. Completed forecast batches are cached locally for restart-safe evaluation.

In [ ]:
from gluonts.itertools import batcher
from gluonts.model.forecast import QuantileForecast

class APIPredictor:
    # Preserve 64-context batches for legacy fp16 forecasts.
    def __init__(self, client, horizon, freq, transport_batch_size=64):
        self.client, self.prediction_length, self.freq = client, horizon, freq
        self.transport_batch_size = transport_batch_size

    def predict(self, test_data_input, batch_size=None):
        for batch in batcher(test_data_input, batch_size=self.transport_batch_size):
            wire_contexts, wire_covariates = [], []
            for entry in batch:
                target = entry['target']
                covariates = entry.get('past_feat_dynamic_real')
                wire_contexts.append(pack_array(target))
                wire_covariates.append(pack_array(covariates))
            req = {'schema_version': 2, 'prediction_length': self.prediction_length,
                   'prediction_freq': self.freq,
                   'contexts': wire_contexts, 'past_only_covariates': wire_covariates}
            response = self.client.predict(req)
            expected_keys = [str(x/10) for x in range(1,10)] + ['mean']
            if response.get('forecast_keys') != expected_keys:
                raise ValueError('API returned wrong protocol or quantile keys')
            arrays = [unpack_array(x) for x in response['forecasts']]
            if len(arrays) != len(batch):
                raise ValueError('API output count mismatch')
            for arr, entry in zip(arrays, batch):
                shape = (len(expected_keys), self.prediction_length)
                if entry['target'].ndim == 2:
                    shape += (entry['target'].shape[0],)
                if arr.shape != shape or not np.isfinite(arr).all():
                    raise ValueError(f'API forecast shape/finite check failed: {arr.shape} expected {shape}')
                yield QuantileForecast(forecast_arrays=arr, forecast_keys=expected_keys,
                    start_date=entry['start'] + entry['target'].shape[-1])


## Full 97-configuration public-cloud evaluation

The configuration list matches the official GIFT-Eval benchmark.

In [ ]:
selected = None
print('Scope: full97')

Each completed logical batch is cached by request hash. Each completed configuration has an atomic row record; on restart the CSV and run JSON are rebuilt from committed rows, then evaluation resumes.

In [ ]:
import csv
import pandas as pd
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality

try:
    from gift_eval.data import Dataset
except ImportError:
    from src.gift_eval.data import Dataset


def atomic_json(path, value):
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(value, indent=2))
    temp.replace(path)


def atomic_csv(path, records):
    temp = path.with_name(path.name + '.tmp')
    with temp.open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(HEADER)
        for record in records:
            writer.writerow(record['row'])
    temp.replace(path)


todo = configurations()
if selected is not None:
    unknown = set(selected) - {row[2] for row in todo}
    if unknown:
        raise ValueError(f'Unknown configurations: {unknown}')
    todo = [row for row in todo if row[2] in selected]


def cost(row):
    ds = Dataset(name=row[0], term=row[1], to_univariate=False)
    return ds.windows * len(ds.hf_dataset) * ds.target_dim * (
        0.0019 + 1.2507e-5 * min(ds._min_series_length, 15360))


todo.sort(key=cost)
selection = [row[2] for row in todo]
evidence_path = csv_file_path.with_suffix('.run.json')
rows_dir = output_dir / 'rows'
rows_dir.mkdir(parents=True, exist_ok=True)
client.cache_dir = output_dir / 'batch_cache'

if evidence_path.exists():
    evidence = json.loads(evidence_path.read_text())
    if (evidence.get('protocol') != 'transport_v2' or
            evidence.get('selection') != selection or
            evidence.get('transport_batch_size') != 64 or
            evidence.get('metric_batch_size') != 64 or
            evidence.get('status') not in ('running', 'failed', 'complete')):
        raise RuntimeError('Run evidence does not match this resumable evaluation')
    if evidence['status'] == 'complete':
        print('This evaluation is already complete; validating committed rows.', flush=True)
else:
    if csv_file_path.exists() or any(rows_dir.iterdir()):
        raise RuntimeError('Output exists without run evidence; refusing to overwrite')
    evidence = {'protocol': 'transport_v2', 'run_id': uuid.uuid4().hex,
                'selection': selection, 'started': time.time(),
                'transport_batch_size': 64, 'metric_batch_size': 64,
                'status': 'running', 'completed': [], 'api_calls': 0,
                'transport_calls': 0, 'cache_hits': 0}
    atomic_json(evidence_path, evidence)

# Row files are the commit log. Rebuild CSV and run evidence after any interruption,
# including a crash between writing a row, CSV, and run.json.
records = []
for index, row in enumerate(todo):
    path = rows_dir / (row[2].replace('/', '__') + '.json')
    if not path.exists():
        if any((rows_dir / (later[2].replace('/', '__') + '.json')).exists()
               for later in todo[index + 1:]):
            raise RuntimeError('Committed row files have a gap')
        break
    record = json.loads(path.read_text())
    if record.get('dataset') != row[2] or record['row'][0] != row[2]:
        raise RuntimeError(f'Invalid committed row: {path}')
    records.append(record)

if evidence['status'] == 'complete' and len(records) != len(todo):
    raise RuntimeError('Completed run is missing committed row files')

evidence['completed'] = [record['completed'] for record in records]
if records:
    last = records[-1]
    evidence['identities'] = last['identities']
    evidence['api_calls'] = last['api_calls']
    evidence['transport_calls'] = last['transport_calls']
    evidence['cache_hits'] = last.get('cache_hits', 0)
    evidence['max_canvas_call_seconds'] = last.get('max_canvas_call_seconds', 0.0)
atomic_csv(csv_file_path, records)
client.run_id = evidence['run_id']
client.identities = evidence.get('identities', {})
client.calls = evidence.get('api_calls', 0)
client.transport_calls = evidence.get('transport_calls', 0)
client.cache_hits = evidence.get('cache_hits', 0)
client.max_canvas_call_seconds = evidence.get('max_canvas_call_seconds', 0.0)

if evidence['status'] != 'complete':
    evidence.pop('error', None)
    evidence.pop('finished', None)
    evidence['status'] = 'running'
    evidence.setdefault('resumed_at', []).append(time.time())
    atomic_json(evidence_path, evidence)

try:
    for name, term, key, freq, prop in todo[len(records):]:
        ds = Dataset(name=name, term=term, to_univariate=False)
        print(f'{key}: windows={len(ds.test_data)}, transport_batch=64', flush=True)
        start = time.time()
        evidence['active'] = key
        atomic_json(evidence_path, evidence)
        res = evaluate_model(
            APIPredictor(client, ds.prediction_length, freq, 64),
            test_data=ds.test_data, metrics=metrics(), batch_size=64,
            axis=None, mask_invalid_label=True, allow_nan_forecast=False,
            seasonality=get_seasonality(ds.freq))
        row = [key, model_name] + [float(res[k][0]) for k in KEYS] + [
            prop['domain'], prop['num_variates']]
        completed = {'dataset': key, 'seconds': time.time() - start}
        record = {'dataset': key, 'row': row, 'completed': completed,
                  'identities': client.identities, 'api_calls': client.calls,
                  'transport_calls': client.transport_calls,
                  'cache_hits': client.cache_hits,
                  'max_canvas_call_seconds': client.max_canvas_call_seconds}
        row_path = rows_dir / (key.replace('/', '__') + '.json')
        atomic_json(row_path, record)
        records.append(record)
        atomic_csv(csv_file_path, records)
        evidence['completed'].append(completed)
        evidence.update(api_calls=client.calls, transport_calls=client.transport_calls,
                        cache_hits=client.cache_hits, identities=client.identities,
                        max_canvas_call_seconds=client.max_canvas_call_seconds)
        evidence.pop('active', None)
        atomic_json(evidence_path, evidence)
    evidence['status'] = 'complete'
except Exception as exc:
    evidence['status'], evidence['error'] = 'failed', str(exc)
    raise
finally:
    evidence['finished'] = time.time()
    atomic_json(evidence_path, evidence)

results = pd.read_csv(csv_file_path)
results
